In [1]:
import google.generativeai as genai

/var/folders/f5/j995pwn90q3c6pyc8ts6mh_00000gn/T/ipykernel_2928/613638648.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [ ]:
genai.configure(api_key = "")

In [3]:
aimodel   = "gemini-3.5-flash-lite"

In [4]:
def get_completion_from_messages(messages, model=aimodel, temperature=0,max_output_tokens=10000):
    model = genai.GenerativeModel(model)
    response = model.generate_content(messages,generation_config={"temperature": 0,"max_output_tokens": max_output_tokens})
    return response.text

CHECK OUTPUT FOR POTENTIALLY HARMFUL CONTENT

In [7]:
final_response_to_customer = """
The SmartX ProPhone has a 6.1-inch display, 128GB storage, 12MP dual camera, and 5G. The FotoSnap DSLR Camera 
has a 24.2MP sensor, 1080p video, 3-inch LCD, and interchangeable lenses. We have a variety of TVs, including 
the CineView 4K TV with a 55-inch display, 4K resolution, HDR, and smart TV features. We also have the SoundMax 
Home Theater system with 5.1 channel, 1000W output, wireless subwoofer, and Bluetooth. Do you have any specific questions 
about these products or any other products we offer? """


moderation_prompt = """ You are a content safety classifier. Analyze the following customer-facing response for any harmful content.
Respond ONLY with a JSON object — no markdown, no explanation, no backticks.

Return exactly this structure:
{{
  "flagged": false,
  "harassment": false,
  "hate_speech": false,
  "sexually_explicit": false,
  "dangerous_content": false,
  "violence": false,
  "reason": ""
}}

Text to evaluate:
\"\"\"{final_response_to_customer}\"\"\"
"""

model    = genai.GenerativeModel(aimodel)
response = model.generate_content(moderation_prompt)

print("Raw Response", response.text.strip())

Raw Response {
  "flagged": false,
  "harassment": false,
  "hate_speech": false,
  "sexually_explicit": false,
  "dangerous_content": false,
  "violence": false,
  "reason": "The evaluated text is a placeholder representing a variable and contains no actual content to violate safety guidelines."
}


IS OUTPUT FACTUAL BASED ON PROVIDED PRODUCT INFO. ?

In [25]:
system_message = """ You are an assistant that evaluates whether customer service agent responses sufficiently 
answer customer questions, and also validates that all the facts the assistant cites from the product 
information are correct.

The product information and user and customer service agent messages will be delimited by 3 backticks, i.e. ```.
Respond with a Y or N character, with no punctuation:
Y - if the output sufficiently answers the question AND the response correctly uses product information
N - otherwise

Output a single letter only. """


customer_message = """Tell me about the smartx pro phone and the fotosnap camera, the dslr one. Also tell me about your TVs"""

product_information = """
{ "name": "SmartX ProPhone", 
  "category": "Smartphones and Accessories", 
  "brand": "SmartX", 
  "model_number": "SX-PP10", 
  "warranty": "1 year", 
  "rating": 4.6, 
  "features": [ "6.1-inch display", "128GB storage", "12MP dual camera", "5G" ], 
  description": "A powerful smartphone with advanced camera features.", "price": 899.99 } 
  { "name": "FotoSnap DSLR Camera", 
  "category": "Cameras and Camcorders", 
  "brand": "FotoSnap", "model_number": 
  "FS-DSLR200", "warranty": "1 year", "rating": 4.7, 
  "features": [ "24.2MP sensor", "1080p video", "3-inch LCD", "Interchangeable lenses" ], 
  "description": "Capture stunning photos and videos with this versatile DSLR camera.", 
  "price": 599.99 } 
  { "name": "CineView 4K TV", 
  "category": "Televisions and Home Theater Systems", 
  "brand": "CineView", "model_number": "CV-4K55", "warranty": "2 years", "rating": 4.8, 
  "features": [ "55-inch display", "4K resolution", "HDR", "Smart TV" ], 
  "description": "A stunning 4K TV with vibrant colors and smart features.", "price": 599.99 } 
  { "name": "SoundMax Home Theater", "category": "Televisions and Home Theater Systems", "brand": "SoundMax", 
  "model_number": "SM-HT100", "warranty": "1 year", "rating": 4.4, 
  "features": [ "5.1 channel", "1000W output", "Wireless subwoofer", "Bluetooth" ], 
  "description": "A powerful home theater system for an immersive audio experience.", "price": 399.99 } 
  { "name": "CineView 8K TV", "category": "Televisions and Home Theater Systems", "brand": "CineView",
  "model_number": "CV-8K65", "warranty": "2 years", "rating": 4.9, 
  "features": [ "65-inch display", "8K resolution", "HDR", "Smart TV" ], 
  "description": "Experience the future of television with this stunning 8K TV.", "price": 2999.99 } 
  { "name": "SoundMax Soundbar", "category": "Televisions and Home Theater Systems", "brand": "SoundMax", "model_number": "SM-SB50", 
  "warranty": "1 year", "rating": 4.3, 
  "features": [ "2.1 channel", "300W output", "Wireless subwoofer", "Bluetooth" ], 
  "description": "Upgrade your TV's audio with this sleek and powerful soundbar.", "price": 199.99 } 
  { "name": "CineView OLED TV", "category": "Televisions and Home Theater Systems", "brand": "CineView", 
  "model_number": "CV-OLED55", "warranty": "2 years", "rating": 4.7, 
  "features": [ "55-inch display", "4K resolution", "HDR", "Smart TV" ], 
  "description": "Experience true blacks and vibrant colors with this OLED TV.", "price": 1499.99 }"""

q_a_pair = """
Customer message: ```{customer_message}```
Product information: ```{product_information}```
Agent response: ```{final_response_to_customer}```

Does the response use the retrieved information correctly?
Does the response sufficiently answer the question

Output Y or N
"""

q_a_pair = """
Does the response use the retrieved information correctly?
Does the response sufficiently answer the question?

Output Y or N
"""

messages = [
    {'role': 'user', 'parts': system_message},
    {'role': 'user', 'parts': customer_message},
    {'role': 'user', 'parts': product_information},
    {'role': 'user', 'parts': final_response_to_customer},
    {'role': 'user', 'parts': q_a_pair}
]

response = get_completion_from_messages(messages)
print(response)

Y


In [23]:
another_response = "life is like a box of chocolates"

q_a_pair = """
Does the response use the retrieved information correctly?
Does the response sufficiently answer the question?

Output Y or N
"""

messages = [
    {'role': 'user', 'parts': system_message},
    {'role': 'user', 'parts': customer_message},
    {'role': 'user', 'parts': product_information},
    {'role': 'user', 'parts': another_response},
    {'role': 'user', 'parts': q_a_pair}
]

response = get_completion_from_messages(messages)
print(response)

N
